# Figure 5Q | Spatial GO programs across the ten MSN anatomical axes

Construction of D1- and D2-specific Gene Ontology (GO) Biological Process libraries, calculation of spatial GO scores, identification of robust WT spatial patterns, and reduction of highly overlapping GO terms to representative functional programs.

The ten analysis groups are:

| Axis | Region | Cell type |
|---|---|---|
| x | Dorsal striatum | MSN_D1 |
| x | Dorsal striatum | MSN_D2 |
| y | Dorsal striatum | MSN_D1 |
| y | Dorsal striatum | MSN_D2 |
| x | NAcc | MSN_D1 |
| x | NAcc | MSN_D2 |
| y | NAcc | MSN_D1 |
| y | NAcc | MSN_D2 |
| x | OT | MSN_D1 |
| x | OT | MSN_D2 |

The uploaded source notebooks processed the GO workflow in four stages. In the original batch scripts, `region + celltype` were used as output identifiers and the anatomical axis was implicit in the upstream matrix set. This release notebook makes `axis + region + celltype` explicit so that all ten analyses can be run together without x/y outputs overwriting one another.

| Stage | Workflow |
|---|---|
| 1 | Build D1/D2 GO Biological Process libraries |
| 2 | Calculate GO scores from gene-level spatial profiles |
| 3 | Identify robust WT spatial-pattern terms |
| 4 | Group highly overlapping GO terms and select representative programs |

Spatial-pattern classification is completed in Stage 3. Stage 4 is a post hoc redundancy-reduction step for figure presentation and does not determine whether a GO term is spatially patterned.


## Input data and GO references

The source analysis used:

- `go-basic.obo`, GO format version 1.2, data release 2026-07-26;
- `MOUSE-mod.gaf.gz`, GAF version 2.2, generated by UniProt on 2026-08-04;
- D1- and D2-specific expressed-gene lists from the higher-plex BASSFISH analysis;
- gene-level spatial matrices generated upstream for each anatomical axis.

Each gene-level spatial matrix contains one gene per row and sample/bin columns named as:

`WT_<sample>_bin<bin>`, `Heter_<sample>_bin<bin>`, or `Homo_<sample>_bin<bin>`.

The matrix values are the previously calculated relative log-ratio spatial profiles. No additional centering is applied in this notebook.


## Imports and configuration


In [ ]:
from pathlib import Path
from collections import defaultdict
from functools import lru_cache
import gzip
import json
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

warnings.filterwarnings("ignore")

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,
    "axes.titlesize": 7,
    "axes.labelsize": 7,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

# ---------------------------------------------------------------------
# Paths: edit for the local repository/data layout.
# ---------------------------------------------------------------------
DATA_ROOT = Path("data")
OUT_ROOT = Path("outputs/Fig5Q_GO_spatial_programs")
FIG_DIR = OUT_ROOT / "figures"

D1_GENE_FILE = DATA_ROOT / "MSN_D1_expressed_genes.txt"
D2_GENE_FILE = DATA_ROOT / "MSN_D2_expressed_genes.txt"

GO_OBO_FILE = DATA_ROOT / "GO_reference/go-basic.obo"
MOUSE_GAF_FILE = DATA_ROOT / "GO_reference/MOUSE-mod.gaf.gz"

# A small manifest that points to the ten upstream spatial-profile matrices.
# Required columns:
# axis, region, celltype, all_samples_matrix_path
MATRIX_MANIFEST = DATA_ROOT / "GO_axis_matrix_manifest.csv"

LIBRARY_DIR = OUT_ROOT / "01_GO_BP_term_library"
SCORE_DIR = OUT_ROOT / "02_GO_BP_term_scores"
PATTERN_DIR = OUT_ROOT / "03_WT_GO_BP_spatial_pattern"
REDUNDANCY_DIR = OUT_ROOT / "04_WT_GO_BP_redundancy_reduction"

for directory in [
    OUT_ROOT,
    FIG_DIR,
    LIBRARY_DIR,
    SCORE_DIR,
    PATTERN_DIR,
    REDUNDANCY_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

REGION_LABELS = {
    "d": "Dorsal striatum",
    "n": "NAcc",
    "o": "OT",
}

# Primary GO-library size limits from the source analysis.
MIN_LIBRARY_GENES = 5
MAX_LIBRARY_GENES = 50

# A term must retain at least five member genes in an axis-specific
# expression matrix before a score is calculated.
MIN_AVAILABLE_GENES = 5

# WT spatial-pattern criteria from the source analysis.
SAMPLE_DIRECTION_CUTOFF = 0.75
R_CUTOFF = 0.50
GENE_DIRECTION_CUTOFF = 0.60
GENE_LOO_CUTOFF = 0.80

# Post hoc GO redundancy reduction.
JACCARD_CUTOFF = 0.50
OVERLAP_COEFF_CUTOFF = 0.80


## Ten-axis matrix manifest

The manifest makes the anatomical axis explicit. It must contain exactly the D1/D2 × Dorsal/NAcc x/y × OT x combinations used in Figure 5Q.

Example:

```text
axis,region,celltype,all_samples_matrix_path
x,d,MSN_D1,...
x,d,MSN_D2,...
y,d,MSN_D1,...
y,d,MSN_D2,...
x,n,MSN_D1,...
x,n,MSN_D2,...
y,n,MSN_D1,...
y,n,MSN_D2,...
x,o,MSN_D1,...
x,o,MSN_D2,...
```


In [ ]:
EXPECTED_GROUPS = pd.DataFrame([
    ("x", "d", "MSN_D1"),
    ("x", "d", "MSN_D2"),
    ("y", "d", "MSN_D1"),
    ("y", "d", "MSN_D2"),
    ("x", "n", "MSN_D1"),
    ("x", "n", "MSN_D2"),
    ("y", "n", "MSN_D1"),
    ("y", "n", "MSN_D2"),
    ("x", "o", "MSN_D1"),
    ("x", "o", "MSN_D2"),
], columns=["axis", "region", "celltype"])

EXPECTED_GROUPS


In [ ]:
def load_matrix_manifest(path):
    path = Path(path)

    if not path.exists():
        template = EXPECTED_GROUPS.copy()
        template["all_samples_matrix_path"] = ""

        print(
            "Matrix manifest not found. Create the following file and fill "
            "the ten matrix paths:"
        )
        print(path)
        display(template)

        raise FileNotFoundError(path)

    manifest = pd.read_csv(path)

    required = {
        "axis",
        "region",
        "celltype",
        "all_samples_matrix_path",
    }

    missing = required - set(manifest.columns)
    if missing:
        raise KeyError(
            f"Manifest is missing required columns: {sorted(missing)}"
        )

    manifest = manifest.copy()
    manifest["axis"] = manifest["axis"].astype(str).str.lower()
    manifest["region"] = manifest["region"].astype(str).str.lower()
    manifest["celltype"] = manifest["celltype"].astype(str)

    observed = set(
        map(
            tuple,
            manifest[["axis", "region", "celltype"]]
            .drop_duplicates()
            .to_numpy(),
        )
    )
    expected = set(map(tuple, EXPECTED_GROUPS.to_numpy()))

    missing_groups = expected - observed
    unexpected_groups = observed - expected

    if missing_groups:
        raise ValueError(
            "Manifest is missing Figure 5Q groups: "
            f"{sorted(missing_groups)}"
        )

    if unexpected_groups:
        print(
            "[NOTE] Additional manifest groups will be ignored:",
            sorted(unexpected_groups),
        )

    manifest = manifest.merge(
        EXPECTED_GROUPS,
        on=["axis", "region", "celltype"],
        how="inner",
    )

    return manifest


# Run after GO_axis_matrix_manifest.csv has been prepared:
# matrix_manifest = load_matrix_manifest(MATRIX_MANIFEST)


# Stage 1 | Construct D1- and D2-specific GO Biological Process libraries

Only active GO Biological Process terms are retained. Alternative GO IDs are resolved to their primary IDs. Mouse direct annotations are propagated to ancestors through `is_a` and `part_of` relationships.

For the GAF annotations:

- only Biological Process (`Aspect = P`) annotations are used;
- annotations carrying the `NOT` qualifier are excluded;
- `ND` evidence is excluded;
- other evidence types are retained.

The root term `GO:0008150` (`biological_process`) is excluded.

The primary term library retains GO terms containing 5–50 measured genes in the corresponding D1 or D2 expressed-gene list.


In [ ]:
def read_gene_list_txt(file_path):
    df = pd.read_csv(
        file_path,
        header=None,
        names=["gene"],
    )

    df["gene"] = (
        df["gene"]
        .astype(str)
        .str.strip()
        .str.strip("'\"")
    )

    df = df[
        ~df["gene"].str.lower().isin(
            ["", "nan", "none", "na", "null"]
        )
    ]

    df = df.drop_duplicates(subset="gene")

    return df["gene"].tolist()


In [ ]:
def parse_go_obo(path):
    terms = {}
    alt_to_primary = {}
    current = None

    def save_current():
        nonlocal current

        if current is None or "id" not in current:
            return

        go_id = current["id"]
        terms[go_id] = current

        for alt_id in current.get("alt_ids", []):
            alt_to_primary[alt_id] = go_id

    with open(path, "r", encoding="utf-8") as f:
        for raw_line in f:
            line = raw_line.strip()

            if line == "[Term]":
                save_current()
                current = {
                    "alt_ids": set(),
                    "parents_is_a": set(),
                    "parents_part_of": set(),
                    "is_obsolete": False,
                }
                continue

            if line.startswith("[") and line != "[Term]":
                save_current()
                current = None
                continue

            if current is None:
                continue

            if line.startswith("id: "):
                current["id"] = line[4:].strip()

            elif line.startswith("name: "):
                current["name"] = line[6:].strip()

            elif line.startswith("namespace: "):
                current["namespace"] = line[11:].strip()

            elif line.startswith("alt_id: "):
                current["alt_ids"].add(
                    line[8:].strip()
                )

            elif line.startswith("is_a: "):
                parent = (
                    line[6:]
                    .split(" ! ")[0]
                    .strip()
                )
                current["parents_is_a"].add(parent)

            elif line.startswith("relationship: part_of "):
                parent = (
                    line
                    .replace("relationship: part_of ", "")
                    .split(" ! ")[0]
                    .strip()
                )
                current["parents_part_of"].add(parent)

            elif line == "is_obsolete: true":
                current["is_obsolete"] = True

        save_current()

    return terms, alt_to_primary


In [ ]:
def build_go_bp_libraries(
    d1_gene_file,
    d2_gene_file,
    go_obo_file,
    mouse_gaf_file,
    outdir,
    min_genes=5,
    max_genes=50,
):
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    D1_genes = read_gene_list_txt(d1_gene_file)
    D2_genes = read_gene_list_txt(d2_gene_file)

    D1_gene_map = {g.casefold(): g for g in D1_genes}
    D2_gene_map = {g.casefold(): g for g in D2_genes}

    go_terms, alt_to_primary = parse_go_obo(go_obo_file)

    bp_terms = {
        go_id: info
        for go_id, info in go_terms.items()
        if (
            info.get("namespace") == "biological_process"
            and not info.get("is_obsolete", False)
        )
    }

    def resolve_go_id(go_id):
        if go_id in go_terms:
            return go_id
        return alt_to_primary.get(go_id)

    @lru_cache(maxsize=None)
    def get_ancestors(go_id):
        go_id = resolve_go_id(go_id)

        if go_id is None:
            return frozenset()

        ancestors = set()
        stack = [go_id]

        while stack:
            current = stack.pop()
            info = go_terms.get(current)

            if info is None:
                continue

            parents = (
                info.get("parents_is_a", set())
                | info.get("parents_part_of", set())
            )

            for parent in parents:
                parent = resolve_go_id(parent)

                if parent is None:
                    continue

                if parent not in ancestors:
                    ancestors.add(parent)
                    stack.append(parent)

        return frozenset(ancestors)

    mouse_gene_direct_go = defaultdict(set)
    n_total = 0
    n_used = 0

    with gzip.open(
        mouse_gaf_file,
        "rt",
        encoding="utf-8",
    ) as f:
        for line in f:
            if line.startswith("!"):
                continue

            cols = line.rstrip("\n").split("\t")

            if len(cols) < 9:
                continue

            n_total += 1

            gene = cols[2].strip()
            qualifier = cols[3].strip()
            go_id = cols[4].strip()
            evidence = cols[6].strip()
            aspect = cols[8].strip()

            if aspect != "P":
                continue

            if "NOT" in qualifier.split("|"):
                continue

            if evidence == "ND":
                continue

            go_id = resolve_go_id(go_id)

            if go_id is None or go_id not in bp_terms:
                continue

            mouse_gene_direct_go[gene].add(go_id)
            n_used += 1

    mouse_gene_map = {
        gene.casefold(): gene
        for gene in mouse_gene_direct_go
    }

    D1_GO_matched = [
        g for g in D1_genes
        if g.casefold() in mouse_gene_map
    ]
    D2_GO_matched = [
        g for g in D2_genes
        if g.casefold() in mouse_gene_map
    ]

    D1_GO_unmatched = [
        g for g in D1_genes
        if g.casefold() not in mouse_gene_map
    ]
    D2_GO_unmatched = [
        g for g in D2_genes
        if g.casefold() not in mouse_gene_map
    ]

    go_to_mouse_genes = defaultdict(set)

    for mouse_gene, direct_terms in mouse_gene_direct_go.items():
        for direct_go in direct_terms:
            related_terms = (
                {direct_go}
                | set(get_ancestors(direct_go))
            )

            for go_id in related_terms:
                if go_id not in bp_terms:
                    continue

                if go_id == "GO:0008150":
                    continue

                go_to_mouse_genes[go_id].add(mouse_gene)

    def make_library(gene_map):
        records = []

        for go_id, mouse_genes in go_to_mouse_genes.items():
            term_genes = {
                gene_map[g.casefold()]
                for g in mouse_genes
                if g.casefold() in gene_map
            }

            if len(term_genes) == 0:
                continue

            database_gene_count = len(mouse_genes)
            gene_count = len(term_genes)

            records.append({
                "term_id": go_id,
                "term_name": bp_terms[go_id]["name"],
                "database_gene_count": database_gene_count,
                "gene_count": gene_count,
                "coverage": (
                    gene_count / database_gene_count
                    if database_gene_count > 0
                    else np.nan
                ),
                "genes": ";".join(sorted(term_genes)),
            })

        all_terms = (
            pd.DataFrame(records)
            .sort_values(
                "gene_count",
                ascending=False,
            )
            .reset_index(drop=True)
        )

        main_terms = all_terms[
            (all_terms["gene_count"] >= min_genes)
            & (all_terms["gene_count"] <= max_genes)
        ].copy()

        return all_terms, main_terms

    D1_all, D1_main = make_library(D1_gene_map)
    D2_all, D2_main = make_library(D2_gene_map)

    D1_all.to_csv(
        outdir / "MSN_D1_GO_BP_all_terms.csv",
        index=False,
    )
    D2_all.to_csv(
        outdir / "MSN_D2_GO_BP_all_terms.csv",
        index=False,
    )
    D1_main.to_csv(
        outdir / "MSN_D1_GO_BP_main_terms.csv",
        index=False,
    )
    D2_main.to_csv(
        outdir / "MSN_D2_GO_BP_main_terms.csv",
        index=False,
    )

    pd.DataFrame({"gene": D1_GO_unmatched}).to_csv(
        outdir / "MSN_D1_GO_unmatched_genes.csv",
        index=False,
    )
    pd.DataFrame({"gene": D2_GO_unmatched}).to_csv(
        outdir / "MSN_D2_GO_unmatched_genes.csv",
        index=False,
    )

    summary = pd.DataFrame({
        "celltype": ["MSN_D1", "MSN_D2"],
        "expressed_genes": [len(D1_genes), len(D2_genes)],
        "GO_matched_genes": [
            len(D1_GO_matched),
            len(D2_GO_matched),
        ],
        "GO_match_rate": [
            len(D1_GO_matched) / len(D1_genes),
            len(D2_GO_matched) / len(D2_genes),
        ],
        "all_GO_BP_terms": [
            len(D1_all),
            len(D2_all),
        ],
        "main_terms_5_50": [
            len(D1_main),
            len(D2_main),
        ],
        "gaf_rows_total": [n_total, n_total],
        "gaf_rows_used": [n_used, n_used],
    })

    summary.to_csv(
        outdir / "GO_BP_library_summary.csv",
        index=False,
    )

    print(summary)

    return {
        "MSN_D1": D1_main,
        "MSN_D2": D2_main,
        "summary": summary,
    }


In [ ]:
# Build the GO libraries once.
#
# go_libraries = build_go_bp_libraries(
#     D1_GENE_FILE,
#     D2_GENE_FILE,
#     GO_OBO_FILE,
#     MOUSE_GAF_FILE,
#     LIBRARY_DIR,
#     min_genes=MIN_LIBRARY_GENES,
#     max_genes=MAX_LIBRARY_GENES,
# )


# Stage 2 | Calculate spatial GO scores for all ten axes

For GO term `t`, sample `s`, and spatial bin `b`, the score is the arithmetic mean of the member-gene spatial profiles available in the corresponding matrix:

**GO_score(t,s,b) = (1 / G_t) × Σ_g X(g,s,b)**

where `G_t` is the number of available member genes and `X(g,s,b)` is the upstream relative log-ratio expression value.

A GO term is skipped for a specific axis if fewer than five of its predefined member genes are available.

A centered 3-bin moving average is calculated only for profile visualization. The raw unsmoothed GO scores are used in Stage 3 to classify spatial patterns.


In [ ]:
def parse_matrix_column(column):
    match = re.match(
        r"^(WT|Heter|Homo)_(.+)_bin(\d+)$",
        str(column),
    )

    if match is None:
        raise ValueError(
            f"Cannot parse spatial matrix column: {column}"
        )

    return {
        "matrix_column": column,
        "condition": match.group(1),
        "sample": match.group(2),
        "bin": int(match.group(3)),
    }


def build_term_profiles(
    matrix_file,
    term_file,
    axis,
    region,
    celltype,
    min_genes=5,
):
    gene_df = pd.read_csv(matrix_file)

    first_col = gene_df.columns[0]
    gene_df = gene_df.rename(
        columns={first_col: "gene"}
    )

    gene_df["gene"] = (
        gene_df["gene"]
        .astype(str)
        .str.strip()
    )

    value_cols = [
        c for c in gene_df.columns
        if c != "gene"
    ]

    col_meta = pd.DataFrame([
        parse_matrix_column(c)
        for c in value_cols
    ])

    term_def = pd.read_csv(term_file)

    gene_idx = gene_df.set_index("gene")
    gene_map = {
        gene.casefold(): gene
        for gene in gene_idx.index
    }

    score_records = []
    qc_records = []

    for _, row in term_def.iterrows():
        defined_genes = [
            gene.strip()
            for gene in str(row["genes"]).split(";")
            if gene.strip()
        ]

        available = []
        missing = []

        for gene in defined_genes:
            key = gene.casefold()

            if key in gene_map:
                available.append(gene_map[key])
            else:
                missing.append(gene)

        status = (
            "ok"
            if len(available) >= min_genes
            else "skip_less_than_min"
        )

        qc_records.append({
            "axis": axis,
            "region": region,
            "celltype": celltype,
            "term_id": row["term_id"],
            "term_name": row["term_name"],
            "defined_gene_count": len(defined_genes),
            "available_gene_count": len(available),
            "missing_gene_count": len(missing),
            "available_genes": ";".join(available),
            "missing_genes": ";".join(missing),
            "status": status,
        })

        if len(available) < min_genes:
            continue

        score = (
            gene_idx
            .loc[available, value_cols]
            .mean(axis=0)
        )

        record = {
            "term_id": row["term_id"],
            "term_name": row["term_name"],
            "gene_count": len(available),
            "genes": ";".join(available),
        }

        record.update(score.to_dict())
        score_records.append(record)

    raw_wide = pd.DataFrame(score_records)
    qc_df = pd.DataFrame(qc_records)

    id_cols = [
        "term_id",
        "term_name",
        "gene_count",
        "genes",
    ]

    long_df = (
        raw_wide
        .melt(
            id_vars=id_cols,
            value_vars=value_cols,
            var_name="matrix_column",
            value_name="score_raw",
        )
        .merge(
            col_meta,
            on="matrix_column",
            how="left",
        )
    )

    long_df["axis"] = axis
    long_df["region"] = region
    long_df["celltype"] = celltype

    long_df = (
        long_df
        .sort_values(
            ["term_id", "sample", "bin"]
        )
        .reset_index(drop=True)
    )

    long_df["score_smooth3"] = (
        long_df
        .groupby(
            ["term_id", "sample"],
            sort=False,
        )["score_raw"]
        .transform(
            lambda x: x.rolling(
                window=3,
                center=True,
                min_periods=1,
            ).mean()
        )
    )

    smooth_wide = (
        long_df
        .pivot_table(
            index=id_cols,
            columns="matrix_column",
            values="score_smooth3",
        )
        .reset_index()
    )

    smooth_wide = smooth_wide[
        id_cols + value_cols
    ]

    condition_summary = (
        long_df
        .groupby(
            [
                "axis",
                "region",
                "celltype",
                "term_id",
                "term_name",
                "gene_count",
                "condition",
                "bin",
            ],
            as_index=False,
        )
        .agg(
            score_raw_mean=("score_raw", "mean"),
            score_raw_sem=("score_raw", "sem"),
            score_smooth3_mean=("score_smooth3", "mean"),
            score_smooth3_sem=("score_smooth3", "sem"),
            n_samples=("sample", "nunique"),
        )
    )

    return {
        "raw_wide": raw_wide,
        "smooth_wide": smooth_wide,
        "long": long_df,
        "condition_summary": condition_summary,
        "qc": qc_df,
        "column_meta": col_meta,
    }


In [ ]:
def resolve_matrix_path(path_string):
    path = Path(str(path_string))

    if path.is_absolute() and path.exists():
        return path

    candidates = [
        path,
        DATA_ROOT / path,
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate

    raise FileNotFoundError(
        f"Cannot find matrix file: {path_string}"
    )


TERM_FILES = {
    "MSN_D1": LIBRARY_DIR / "MSN_D1_GO_BP_main_terms.csv",
    "MSN_D2": LIBRARY_DIR / "MSN_D2_GO_BP_main_terms.csv",
}


In [ ]:
def run_all_term_scores(matrix_manifest):
    batch_records = []

    for _, info in matrix_manifest.iterrows():
        axis = str(info["axis"])
        region = str(info["region"])
        celltype = str(info["celltype"])

        matrix_file = resolve_matrix_path(
            info["all_samples_matrix_path"]
        )
        term_file = TERM_FILES[celltype]

        print(
            "\n" + "=" * 80,
            f"\naxis_{axis} | region_{region} | {celltype}",
        )

        result = build_term_profiles(
            matrix_file=matrix_file,
            term_file=term_file,
            axis=axis,
            region=region,
            celltype=celltype,
            min_genes=MIN_AVAILABLE_GENES,
        )

        group_outdir = (
            SCORE_DIR
            / f"axis_{axis}"
            / f"region_{region}"
            / f"celltype_{celltype}"
        )
        group_outdir.mkdir(
            parents=True,
            exist_ok=True,
        )

        prefix = (
            f"axis_{axis}_region_{region}_"
            f"celltype_{celltype}"
        )

        result["raw_wide"].to_csv(
            group_outdir
            / f"{prefix}_term_sample_profiles_raw_wide.csv",
            index=False,
        )
        result["smooth_wide"].to_csv(
            group_outdir
            / f"{prefix}_term_sample_profiles_smooth3_wide.csv",
            index=False,
        )
        result["long"].to_csv(
            group_outdir
            / f"{prefix}_term_sample_profiles_long.csv",
            index=False,
        )
        result["condition_summary"].to_csv(
            group_outdir
            / f"{prefix}_term_condition_profiles_summary.csv",
            index=False,
        )
        result["qc"].to_csv(
            group_outdir
            / f"{prefix}_term_member_QC.csv",
            index=False,
        )

        qc = result["qc"]

        batch_records.append({
            "axis": axis,
            "region": region,
            "celltype": celltype,
            "input_terms": len(qc),
            "calculated_terms": int(
                (qc["status"] == "ok").sum()
            ),
            "skipped_terms": int(
                (qc["status"] != "ok").sum()
            ),
            "max_missing_genes": (
                qc["missing_gene_count"].max()
            ),
        })

    batch_summary = pd.DataFrame(batch_records)

    batch_summary.to_csv(
        SCORE_DIR / "term_score_batch_summary.csv",
        index=False,
    )

    return batch_summary


# Run after Stage 1 and after loading the ten-axis manifest:
#
# matrix_manifest = load_matrix_manifest(MATRIX_MANIFEST)
# term_score_summary = run_all_term_scores(matrix_manifest)
# term_score_summary


# Stage 3 | Identify robust WT spatial-pattern terms

WT mice define the reference spatial pattern.

For each GO term:

1. within each WT sample, the ten bins are internally rescaled to 0–1 and a linear slope (`delta_fit`) is fitted to the raw GO score;
2. the mean slope across WT samples defines the overall direction;
3. sample direction consistency is the fraction of WT samples whose slope has the same sign as the mean slope;
4. the WT mean raw profile is correlated with spatial position, and `directional_r = |r|`;
5. each member gene receives a WT mean spatial slope;
6. gene direction consistency is the fraction of member genes whose mean slope matches the GO-term direction;
7. gene leave-one-out stability is the fraction of single-gene removals that preserve the GO-term direction.

A GO term is classified as a robust WT spatial pattern only when all four hard criteria are satisfied:

- sample direction consistency ≥ 0.75;
- directional |r| ≥ 0.50;
- gene direction consistency ≥ 0.60;
- gene leave-one-out direction stability ≥ 0.80.

Positive mean slope is classified as `low_to_high`; negative mean slope as `high_to_low`.

Sample leave-one-out direction stability, maximum single-gene slope contribution, and median pairwise member-gene correlation are retained as QC metrics but are not hard filters.


In [ ]:
def calculate_spatial_trend(bins, values):
    bins = np.asarray(bins, dtype=float)
    values = np.asarray(values, dtype=float)

    valid = (
        np.isfinite(bins)
        & np.isfinite(values)
    )

    bins = bins[valid]
    values = values[valid]

    if len(values) < 3:
        return {
            "delta_fit": np.nan,
            "r": np.nan,
            "dynamic_range": np.nan,
        }

    if bins.max() == bins.min():
        return {
            "delta_fit": np.nan,
            "r": np.nan,
            "dynamic_range": np.nan,
        }

    x = (
        (bins - bins.min())
        / (bins.max() - bins.min())
    )

    delta_fit, intercept = np.polyfit(
        x,
        values,
        1,
    )

    if np.std(values) == 0:
        r = 0.0
    else:
        r = np.corrcoef(x, values)[0, 1]

    dynamic_range = (
        values.max() - values.min()
    )

    return {
        "delta_fit": delta_fit,
        "r": r,
        "dynamic_range": dynamic_range,
    }


def parse_matrix_columns(columns):
    records = []

    for col in columns:
        match = re.match(
            r"^(WT|Heter|Homo)_(.+)_bin(\d+)$",
            str(col),
        )

        if match is None:
            continue

        records.append({
            "matrix_column": col,
            "condition": match.group(1),
            "sample": match.group(2),
            "bin": int(match.group(3)),
        })

    return pd.DataFrame(records)


In [ ]:
def analyze_WT_spatial_terms(
    axis,
    region,
    celltype,
    term_long_file,
    gene_matrix_file,
):
    term_long = pd.read_csv(term_long_file)

    required_columns = [
        "term_id",
        "term_name",
        "gene_count",
        "genes",
        "condition",
        "sample",
        "bin",
        "score_raw",
        "score_smooth3",
    ]

    missing_columns = [
        c for c in required_columns
        if c not in term_long.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Term-score table is missing: {missing_columns}"
        )

    wt_term = term_long[
        term_long["condition"] == "WT"
    ].copy()

    # ---------------------------------------------------------
    # Sample-level term slopes.
    # ---------------------------------------------------------
    sample_trend_records = []

    for (
        term_id,
        term_name,
        sample,
    ), sub in wt_term.groupby(
        ["term_id", "term_name", "sample"]
    ):
        sub = sub.sort_values("bin")

        stats = calculate_spatial_trend(
            sub["bin"],
            sub["score_raw"],
        )

        sample_trend_records.append({
            "axis": axis,
            "region": region,
            "celltype": celltype,
            "term_id": term_id,
            "term_name": term_name,
            "sample": sample,
            "gene_count": int(
                sub["gene_count"].iloc[0]
            ),
            "delta_fit": stats["delta_fit"],
            "r": stats["r"],
            "dynamic_range": stats["dynamic_range"],
        })

    sample_trend_df = pd.DataFrame(
        sample_trend_records
    )

    # ---------------------------------------------------------
    # WT mean term profile.
    # ---------------------------------------------------------
    wt_mean_profile = (
        wt_term
        .groupby(
            ["term_id", "term_name", "bin"],
            as_index=False,
        )
        .agg(
            score_raw_mean=("score_raw", "mean"),
            score_smooth3_mean=(
                "score_smooth3",
                "mean",
            ),
        )
    )

    mean_profile_records = []

    for (
        term_id,
        term_name,
    ), sub in wt_mean_profile.groupby(
        ["term_id", "term_name"]
    ):
        sub = sub.sort_values("bin")

        stats = calculate_spatial_trend(
            sub["bin"],
            sub["score_raw_mean"],
        )

        mean_profile_records.append({
            "term_id": term_id,
            "wt_mean_profile_delta_fit": (
                stats["delta_fit"]
            ),
            "wt_mean_profile_r": stats["r"],
            "directional_r": abs(stats["r"]),
            "wt_mean_profile_dynamic_range": (
                stats["dynamic_range"]
            ),
        })

    mean_profile_stats = pd.DataFrame(
        mean_profile_records
    )

    # ---------------------------------------------------------
    # Sample direction consistency.
    # ---------------------------------------------------------
    summary_records = []

    for (
        term_id,
        term_name,
    ), sub in sample_trend_df.groupby(
        ["term_id", "term_name"]
    ):
        deltas = (
            sub["delta_fit"]
            .dropna()
            .to_numpy(dtype=float)
        )

        if len(deltas) == 0:
            continue

        mean_delta = np.mean(deltas)

        if mean_delta > 0:
            sample_direction_consistency = np.mean(
                deltas > 0
            )
        elif mean_delta < 0:
            sample_direction_consistency = np.mean(
                deltas < 0
            )
        else:
            sample_direction_consistency = 0

        summary_records.append({
            "axis": axis,
            "region": region,
            "celltype": celltype,
            "term_id": term_id,
            "term_name": term_name,
            "gene_count": int(
                sub["gene_count"].iloc[0]
            ),
            "n_WT_samples": len(deltas),
            "mean_delta_fit": mean_delta,
            "median_delta_fit": np.median(deltas),
            "sd_delta_fit": (
                np.std(deltas, ddof=1)
                if len(deltas) > 1
                else np.nan
            ),
            "mean_sample_r": sub["r"].mean(),
            "mean_sample_dynamic_range": (
                sub["dynamic_range"].mean()
            ),
            "sample_direction_consistency": (
                sample_direction_consistency
            ),
        })

    wt_summary = pd.DataFrame(summary_records)

    wt_summary = wt_summary.merge(
        mean_profile_stats,
        on="term_id",
        how="left",
    )

    # ---------------------------------------------------------
    # Sample leave-one-out: QC only.
    # ---------------------------------------------------------
    sample_loo_records = []

    for term_id, sub in sample_trend_df.groupby(
        "term_id"
    ):
        deltas = (
            sub["delta_fit"]
            .dropna()
            .to_numpy(dtype=float)
        )

        if len(deltas) < 2:
            continue

        full_mean = np.mean(deltas)

        loo_means = np.asarray([
            np.mean(np.delete(deltas, i))
            for i in range(len(deltas))
        ])

        if full_mean > 0:
            stability = np.mean(loo_means > 0)
        elif full_mean < 0:
            stability = np.mean(loo_means < 0)
        else:
            stability = 0

        sample_loo_records.append({
            "term_id": term_id,
            "sample_loo_direction_stability": stability,
            "sample_loo_min_delta": np.min(loo_means),
            "sample_loo_max_delta": np.max(loo_means),
        })

    wt_summary = wt_summary.merge(
        pd.DataFrame(sample_loo_records),
        on="term_id",
        how="left",
    )

    # ---------------------------------------------------------
    # Gene-level WT slopes.
    # ---------------------------------------------------------
    gene_df = pd.read_csv(gene_matrix_file)

    first_col = gene_df.columns[0]
    gene_df = gene_df.rename(
        columns={first_col: "gene"}
    )
    gene_df["gene"] = (
        gene_df["gene"]
        .astype(str)
        .str.strip()
    )
    gene_df = gene_df.set_index("gene")

    column_meta = parse_matrix_columns(
        gene_df.columns
    )

    wt_gene_meta = column_meta[
        column_meta["condition"] == "WT"
    ].copy()

    gene_sample_records = []

    for sample, meta in wt_gene_meta.groupby(
        "sample"
    ):
        meta = meta.sort_values("bin")
        cols = meta["matrix_column"].tolist()
        bins = meta["bin"].to_numpy()

        for gene in gene_df.index:
            values = (
                gene_df.loc[gene, cols]
                .to_numpy(dtype=float)
            )

            stats = calculate_spatial_trend(
                bins,
                values,
            )

            gene_sample_records.append({
                "gene": gene,
                "sample": sample,
                "delta_fit": stats["delta_fit"],
            })

    gene_sample_trends = pd.DataFrame(
        gene_sample_records
    )

    gene_mean_delta = (
        gene_sample_trends
        .groupby("gene")["delta_fit"]
        .mean()
    )
    gene_delta_dict = gene_mean_delta.to_dict()

    term_gene_info = (
        wt_term[
            [
                "term_id",
                "term_name",
                "gene_count",
                "genes",
            ]
        ]
        .drop_duplicates(subset="term_id")
    )

    term_direction_dict = (
        wt_summary
        .set_index("term_id")["mean_delta_fit"]
        .to_dict()
    )

    gene_qc_records = []

    for _, row in term_gene_info.iterrows():
        term_id = row["term_id"]

        genes = [
            g.strip()
            for g in str(row["genes"]).split(";")
            if g.strip()
        ]

        available_genes = [
            g for g in genes
            if g in gene_delta_dict
        ]

        gene_deltas = np.asarray(
            [
                gene_delta_dict[g]
                for g in available_genes
            ],
            dtype=float,
        )

        if len(gene_deltas) < 2:
            continue

        term_delta = term_direction_dict[term_id]

        if term_delta > 0:
            gene_direction_consistency = np.mean(
                gene_deltas > 0
            )
        elif term_delta < 0:
            gene_direction_consistency = np.mean(
                gene_deltas < 0
            )
        else:
            gene_direction_consistency = 0

        total_delta = np.sum(gene_deltas)
        n_genes = len(gene_deltas)

        loo_deltas = (
            total_delta - gene_deltas
        ) / (n_genes - 1)

        if term_delta > 0:
            gene_loo_stability = np.mean(
                loo_deltas > 0
            )
        elif term_delta < 0:
            gene_loo_stability = np.mean(
                loo_deltas < 0
            )
        else:
            gene_loo_stability = 0

        abs_sum = np.sum(
            np.abs(gene_deltas)
        )

        max_gene_contribution = (
            np.max(np.abs(gene_deltas)) / abs_sum
            if abs_sum > 0
            else np.nan
        )

        gene_qc_records.append({
            "term_id": term_id,
            "available_gene_count_QC": n_genes,
            "gene_direction_consistency": (
                gene_direction_consistency
            ),
            "gene_loo_direction_stability": (
                gene_loo_stability
            ),
            "max_abs_gene_contribution": (
                max_gene_contribution
            ),
        })

    wt_summary = wt_summary.merge(
        pd.DataFrame(gene_qc_records),
        on="term_id",
        how="left",
    )

    # ---------------------------------------------------------
    # Median pairwise gene correlation: QC only.
    # ---------------------------------------------------------
    all_bins = sorted(
        wt_gene_meta["bin"].unique()
    )

    gene_wt_mean_profile = pd.DataFrame(
        index=gene_df.index,
        columns=all_bins,
        dtype=float,
    )

    for bin_id in all_bins:
        cols = wt_gene_meta.loc[
            wt_gene_meta["bin"] == bin_id,
            "matrix_column",
        ].tolist()

        gene_wt_mean_profile[bin_id] = (
            gene_df[cols].mean(axis=1)
        )

    gene_corr = (
        gene_wt_mean_profile
        .T
        .corr()
    )

    corr_records = []

    for _, row in term_gene_info.iterrows():
        genes = [
            g.strip()
            for g in str(row["genes"]).split(";")
            if (
                g.strip()
                and g.strip() in gene_corr.index
            )
        ]

        if len(genes) < 2:
            median_corr = np.nan
        else:
            corr_matrix = (
                gene_corr
                .loc[genes, genes]
                .to_numpy()
            )
            upper_idx = np.triu_indices(
                len(genes),
                k=1,
            )
            corr_values = corr_matrix[upper_idx]
            corr_values = corr_values[
                np.isfinite(corr_values)
            ]

            median_corr = (
                np.median(corr_values)
                if len(corr_values) > 0
                else np.nan
            )

        corr_records.append({
            "term_id": row["term_id"],
            "median_gene_pairwise_corr": (
                median_corr
            ),
        })

    wt_summary = wt_summary.merge(
        pd.DataFrame(corr_records),
        on="term_id",
        how="left",
    )

    # ---------------------------------------------------------
    # Hard spatial-pattern criteria.
    # ---------------------------------------------------------
    wt_summary["pass_sample_direction"] = (
        wt_summary[
            "sample_direction_consistency"
        ] >= SAMPLE_DIRECTION_CUTOFF
    )

    wt_summary["pass_directional_r"] = (
        wt_summary["directional_r"]
        >= R_CUTOFF
    )

    wt_summary["pass_gene_direction"] = (
        wt_summary[
            "gene_direction_consistency"
        ] >= GENE_DIRECTION_CUTOFF
    )

    wt_summary["pass_gene_loo"] = (
        wt_summary[
            "gene_loo_direction_stability"
        ] >= GENE_LOO_CUTOFF
    )

    wt_summary["is_spatial_pattern"] = (
        wt_summary["pass_sample_direction"]
        & wt_summary["pass_directional_r"]
        & wt_summary["pass_gene_direction"]
        & wt_summary["pass_gene_loo"]
    )

    def assign_category(row):
        if not row["is_spatial_pattern"]:
            return "non_gradient"

        if row["mean_delta_fit"] > 0:
            return "low_to_high"

        if row["mean_delta_fit"] < 0:
            return "high_to_low"

        return "non_gradient"

    wt_summary["spatial_category"] = (
        wt_summary.apply(
            assign_category,
            axis=1,
        )
    )

    wt_summary["pattern_strength"] = (
        wt_summary["mean_delta_fit"].abs()
    )

    # ---------------------------------------------------------
    # Sequential filter summary.
    # ---------------------------------------------------------
    n_total = len(wt_summary)

    step1 = wt_summary[
        "pass_sample_direction"
    ]
    step2 = (
        step1
        & wt_summary["pass_directional_r"]
    )
    step3 = (
        step2
        & wt_summary["pass_gene_direction"]
    )
    step4 = (
        step3
        & wt_summary["pass_gene_loo"]
    )

    filter_summary = pd.DataFrame({
        "axis": [axis] * 5,
        "region": [region] * 5,
        "celltype": [celltype] * 5,
        "step": [
            "All terms",
            (
                "Sample direction >= "
                f"{SAMPLE_DIRECTION_CUTOFF}"
            ),
            f"+ directional r >= {R_CUTOFF}",
            (
                "+ gene direction >= "
                f"{GENE_DIRECTION_CUTOFF}"
            ),
            f"+ gene LOO >= {GENE_LOO_CUTOFF}",
        ],
        "n_terms": [
            n_total,
            int(step1.sum()),
            int(step2.sum()),
            int(step3.sum()),
            int(step4.sum()),
        ],
    })

    filter_summary["fraction_of_all"] = (
        filter_summary["n_terms"]
        / n_total
    )

    category_counts = (
        wt_summary["spatial_category"]
        .value_counts()
        .rename_axis("spatial_category")
        .reset_index(name="n_terms")
    )
    category_counts["axis"] = axis
    category_counts["region"] = region
    category_counts["celltype"] = celltype

    low_to_high = (
        wt_summary[
            wt_summary["spatial_category"]
            == "low_to_high"
        ]
        .sort_values(
            "mean_delta_fit",
            ascending=False,
        )
        .copy()
    )

    high_to_low = (
        wt_summary[
            wt_summary["spatial_category"]
            == "high_to_low"
        ]
        .sort_values(
            "mean_delta_fit",
            ascending=True,
        )
        .copy()
    )

    spatial_ids = wt_summary.loc[
        wt_summary["is_spatial_pattern"],
        "term_id",
    ].tolist()

    spatial_profile = wt_mean_profile[
        wt_mean_profile["term_id"].isin(
            spatial_ids
        )
    ].copy()

    if len(spatial_ids) > 0:
        heatmap_raw = (
            spatial_profile
            .pivot(
                index="term_id",
                columns="bin",
                values="score_smooth3_mean",
            )
        )

        ordered_term_ids = (
            high_to_low["term_id"].tolist()
            + low_to_high["term_id"].tolist()
        )

        heatmap_raw = heatmap_raw.reindex(
            ordered_term_ids
        )

        row_mean = heatmap_raw.mean(axis=1)
        row_std = (
            heatmap_raw
            .std(axis=1)
            .replace(0, np.nan)
        )

        heatmap_z = (
            heatmap_raw
            .sub(row_mean, axis=0)
            .div(row_std, axis=0)
        )
    else:
        heatmap_raw = pd.DataFrame()
        heatmap_z = pd.DataFrame()

    return {
        "sample_trends": sample_trend_df,
        "wt_summary": wt_summary,
        "filter_summary": filter_summary,
        "category_counts": category_counts,
        "low_to_high": low_to_high,
        "high_to_low": high_to_low,
        "heatmap_raw": heatmap_raw,
        "heatmap_z": heatmap_z,
        "wt_mean_profile": wt_mean_profile,
    }


In [ ]:
def run_all_WT_spatial_patterns(matrix_manifest):
    group_summaries = []
    filter_summaries = []
    category_summaries = []

    for _, info in matrix_manifest.iterrows():
        axis = str(info["axis"])
        region = str(info["region"])
        celltype = str(info["celltype"])

        matrix_file = resolve_matrix_path(
            info["all_samples_matrix_path"]
        )

        score_group_dir = (
            SCORE_DIR
            / f"axis_{axis}"
            / f"region_{region}"
            / f"celltype_{celltype}"
        )

        prefix = (
            f"axis_{axis}_region_{region}_"
            f"celltype_{celltype}"
        )

        term_long_file = (
            score_group_dir
            / f"{prefix}_term_sample_profiles_long.csv"
        )

        result = analyze_WT_spatial_terms(
            axis=axis,
            region=region,
            celltype=celltype,
            term_long_file=term_long_file,
            gene_matrix_file=matrix_file,
        )

        group_outdir = (
            PATTERN_DIR
            / f"axis_{axis}"
            / f"region_{region}"
            / f"celltype_{celltype}"
        )
        group_outdir.mkdir(
            parents=True,
            exist_ok=True,
        )

        result["sample_trends"].to_csv(
            group_outdir
            / f"{prefix}_WT_term_sample_trends.csv",
            index=False,
        )
        result["wt_summary"].to_csv(
            group_outdir
            / f"{prefix}_WT_term_spatial_pattern_statistics.csv",
            index=False,
        )
        result["filter_summary"].to_csv(
            group_outdir
            / f"{prefix}_WT_filter_summary.csv",
            index=False,
        )
        result["category_counts"].to_csv(
            group_outdir
            / f"{prefix}_WT_spatial_category_counts.csv",
            index=False,
        )
        result["low_to_high"].to_csv(
            group_outdir
            / f"{prefix}_WT_low_to_high_terms.csv",
            index=False,
        )
        result["high_to_low"].to_csv(
            group_outdir
            / f"{prefix}_WT_high_to_low_terms.csv",
            index=False,
        )
        result["wt_mean_profile"].to_csv(
            group_outdir
            / f"{prefix}_WT_term_mean_profiles.csv",
            index=False,
        )

        if not result["heatmap_raw"].empty:
            result["heatmap_raw"].to_csv(
                group_outdir
                / f"{prefix}_WT_spatial_term_heatmap_raw.csv"
            )
            result["heatmap_z"].to_csv(
                group_outdir
                / f"{prefix}_WT_spatial_term_heatmap_zscore.csv"
            )

        summary = result["wt_summary"]

        group_summaries.append({
            "axis": axis,
            "region": region,
            "celltype": celltype,
            "all_terms": len(summary),
            "low_to_high": int(
                (
                    summary["spatial_category"]
                    == "low_to_high"
                ).sum()
            ),
            "high_to_low": int(
                (
                    summary["spatial_category"]
                    == "high_to_low"
                ).sum()
            ),
            "non_gradient": int(
                (
                    summary["spatial_category"]
                    == "non_gradient"
                ).sum()
            ),
            "spatial_total": int(
                summary["is_spatial_pattern"].sum()
            ),
            "spatial_fraction": (
                summary[
                    "is_spatial_pattern"
                ].mean()
            ),
        })

        filter_summaries.append(
            result["filter_summary"]
        )
        category_summaries.append(
            result["category_counts"]
        )

    group_summary_df = pd.DataFrame(
        group_summaries
    )

    all_filter_df = pd.concat(
        filter_summaries,
        ignore_index=True,
    )

    all_category_df = pd.concat(
        category_summaries,
        ignore_index=True,
    )

    group_summary_df.to_csv(
        PATTERN_DIR
        / "all_axis_region_celltype_WT_spatial_pattern_summary.csv",
        index=False,
    )
    all_filter_df.to_csv(
        PATTERN_DIR
        / "all_axis_region_celltype_filter_summary.csv",
        index=False,
    )
    all_category_df.to_csv(
        PATTERN_DIR
        / "all_axis_region_celltype_spatial_category_counts.csv",
        index=False,
    )

    return (
        group_summary_df,
        all_filter_df,
        all_category_df,
    )


# Run after Stage 2:
#
# wt_pattern_summary, wt_filter_summary, wt_category_summary = (
#     run_all_WT_spatial_patterns(matrix_manifest)
# )
#
# wt_pattern_summary


# Stage 4 | Reduce redundant spatial GO terms

This stage is applied only after the robust WT spatial terms have been defined.

High-to-low and low-to-high terms are processed separately. Two GO terms are treated as a redundant pair when either:

- Jaccard similarity of their library gene sets ≥ 0.50; or
- overlap coefficient ≥ 0.80.

A complete-linkage hierarchy is built from the resulting binary redundant/non-redundant distance matrix. One representative term is selected per redundancy cluster using the ranking recorded in the source notebook:

1. sample leave-one-out direction stability;
2. sample direction consistency;
3. gene leave-one-out direction stability;
4. gene direction consistency;
5. directional `|r|`;
6. pattern strength (`|mean_delta_fit|`);
7. smaller database gene count.

The resulting representative terms are used for compact functional-program visualization. This step does not change the Stage 3 spatial-pattern classification.


In [ ]:
def gene_set_similarity(genes_a, genes_b):
    a = set(genes_a)
    b = set(genes_b)

    if len(a) == 0 or len(b) == 0:
        return {
            "intersection": 0,
            "union": len(a | b),
            "jaccard": 0.0,
            "overlap_coefficient": 0.0,
        }

    intersection = len(a & b)
    union = len(a | b)

    jaccard = (
        intersection / union
        if union > 0
        else 0.0
    )

    overlap_coefficient = (
        intersection / min(len(a), len(b))
    )

    return {
        "intersection": intersection,
        "union": union,
        "jaccard": jaccard,
        "overlap_coefficient": overlap_coefficient,
    }


def is_redundant_pair(
    genes_a,
    genes_b,
    jaccard_cutoff=0.50,
    overlap_cutoff=0.80,
):
    sim = gene_set_similarity(
        genes_a,
        genes_b,
    )

    redundant = (
        sim["jaccard"] >= jaccard_cutoff
        or sim["overlap_coefficient"]
        >= overlap_cutoff
    )

    return redundant, sim


In [ ]:
def load_term_library(celltype):
    term_file = TERM_FILES[celltype]
    df = pd.read_csv(term_file)

    gene_sets = {}

    for _, row in df.iterrows():
        genes = [
            gene.strip()
            for gene in str(row["genes"]).split(";")
            if gene.strip()
        ]

        gene_sets[row["term_id"]] = set(genes)

    return df, gene_sets


def cluster_redundant_terms(
    spatial_df,
    gene_sets,
    jaccard_cutoff=0.50,
    overlap_cutoff=0.80,
):
    term_ids = spatial_df["term_id"].tolist()
    n = len(term_ids)

    if n == 0:
        return pd.DataFrame(), pd.DataFrame()

    if n == 1:
        cluster_df = spatial_df[["term_id"]].copy()
        cluster_df["redundancy_cluster"] = 1
        return cluster_df, pd.DataFrame()

    distance_matrix = np.ones(
        (n, n),
        dtype=float,
    )
    np.fill_diagonal(distance_matrix, 0)

    pair_records = []

    for i in range(n):
        for j in range(i + 1, n):
            term_i = term_ids[i]
            term_j = term_ids[j]

            redundant, sim = is_redundant_pair(
                gene_sets[term_i],
                gene_sets[term_j],
                jaccard_cutoff=jaccard_cutoff,
                overlap_cutoff=overlap_cutoff,
            )

            if redundant:
                distance_matrix[i, j] = 0
                distance_matrix[j, i] = 0

            pair_records.append({
                "term_id_1": term_i,
                "term_id_2": term_j,
                "intersection_gene_count": (
                    sim["intersection"]
                ),
                "jaccard": sim["jaccard"],
                "overlap_coefficient": (
                    sim["overlap_coefficient"]
                ),
                "is_redundant_pair": redundant,
            })

    condensed = squareform(
        distance_matrix,
        checks=False,
    )

    Z = linkage(
        condensed,
        method="complete",
    )

    cluster_labels = fcluster(
        Z,
        t=0.5,
        criterion="distance",
    )

    cluster_df = pd.DataFrame({
        "term_id": term_ids,
        "redundancy_cluster": cluster_labels,
    })

    return (
        cluster_df,
        pd.DataFrame(pair_records),
    )


In [ ]:
def select_representative_terms(clustered_df):
    result_records = []

    rank_cols = [
        "sample_loo_direction_stability",
        "sample_direction_consistency",
        "gene_loo_direction_stability",
        "gene_direction_consistency",
        "directional_r",
        "pattern_strength",
        "database_gene_count",
    ]

    ascending = [
        False,
        False,
        False,
        False,
        False,
        False,
        True,
    ]

    for cluster_id, sub in clustered_df.groupby(
        "redundancy_cluster"
    ):
        ranked = (
            sub
            .sort_values(
                by=rank_cols,
                ascending=ascending,
            )
            .copy()
        )

        ranked["candidate_rank"] = np.arange(
            1,
            len(ranked) + 1,
        )

        representative = ranked.iloc[0]

        for _, row in ranked.iterrows():
            record = row.to_dict()

            record["is_representative"] = (
                row["term_id"]
                == representative["term_id"]
            )
            record["representative_term_id"] = (
                representative["term_id"]
            )
            record["representative_term_name"] = (
                representative["term_name"]
            )

            result_records.append(record)

    return pd.DataFrame(result_records)


In [ ]:
def reduce_one_direction(
    stats_df,
    term_library_df,
    gene_sets,
    spatial_category,
):
    spatial_df = stats_df[
        stats_df["spatial_category"]
        == spatial_category
    ].copy()

    if len(spatial_df) == 0:
        return {
            "clustered": pd.DataFrame(),
            "representatives": pd.DataFrame(),
            "pairwise": pd.DataFrame(),
            "cluster_summary": pd.DataFrame(),
        }

    library_cols = [
        "term_id",
        "database_gene_count",
        "gene_count",
        "coverage",
        "genes",
    ]

    available_cols = [
        c for c in library_cols
        if c in term_library_df.columns
    ]

    spatial_df = spatial_df.merge(
        term_library_df[available_cols],
        on="term_id",
        how="left",
        suffixes=("", "_library"),
    )

    cluster_map, pair_df = cluster_redundant_terms(
        spatial_df=spatial_df,
        gene_sets=gene_sets,
        jaccard_cutoff=JACCARD_CUTOFF,
        overlap_cutoff=OVERLAP_COEFF_CUTOFF,
    )

    clustered = spatial_df.merge(
        cluster_map,
        on="term_id",
        how="left",
    )

    clustered = select_representative_terms(
        clustered
    )

    representatives = clustered[
        clustered["is_representative"]
    ].copy()

    cluster_records = []

    for cluster_id, sub in clustered.groupby(
        "redundancy_cluster"
    ):
        rep = sub[
            sub["is_representative"]
        ].iloc[0]

        cluster_records.append({
            "spatial_category": spatial_category,
            "redundancy_cluster": cluster_id,
            "n_terms": len(sub),
            "representative_term_id": (
                rep["term_id"]
            ),
            "representative_term_name": (
                rep["term_name"]
            ),
            "cluster_term_ids": ";".join(
                sub["term_id"].tolist()
            ),
            "cluster_term_names": " | ".join(
                sub["term_name"].tolist()
            ),
        })

    return {
        "clustered": clustered,
        "representatives": representatives,
        "pairwise": pair_df,
        "cluster_summary": pd.DataFrame(
            cluster_records
        ),
    }


In [ ]:
def analyze_redundancy_group(
    axis,
    region,
    celltype,
):
    prefix = (
        f"axis_{axis}_region_{region}_"
        f"celltype_{celltype}"
    )

    group_dir = (
        PATTERN_DIR
        / f"axis_{axis}"
        / f"region_{region}"
        / f"celltype_{celltype}"
    )

    stats_file = (
        group_dir
        / f"{prefix}_WT_term_spatial_pattern_statistics.csv"
    )

    profile_file = (
        group_dir
        / f"{prefix}_WT_term_mean_profiles.csv"
    )

    stats_df = pd.read_csv(stats_file)
    mean_profile_df = pd.read_csv(profile_file)

    term_library_df, gene_sets = (
        load_term_library(celltype)
    )

    low_result = reduce_one_direction(
        stats_df,
        term_library_df,
        gene_sets,
        "low_to_high",
    )

    high_result = reduce_one_direction(
        stats_df,
        term_library_df,
        gene_sets,
        "high_to_low",
    )

    clustered_all = pd.concat(
        [
            high_result["clustered"],
            low_result["clustered"],
        ],
        ignore_index=True,
    )

    representatives = pd.concat(
        [
            high_result["representatives"],
            low_result["representatives"],
        ],
        ignore_index=True,
    )

    cluster_summary = pd.concat(
        [
            high_result["cluster_summary"],
            low_result["cluster_summary"],
        ],
        ignore_index=True,
    )

    if len(cluster_summary) > 0:
        cluster_summary[
            "functional_program_id"
        ] = [
            (
                f"{category}_"
                f"cluster_{i + 1:03d}"
            )
            for i, category in enumerate(
                cluster_summary[
                    "spatial_category"
                ]
            )
        ]

    high_rep = (
        representatives[
            representatives["spatial_category"]
            == "high_to_low"
        ]
        .sort_values(
            "mean_delta_fit",
            ascending=True,
        )
    )

    low_rep = (
        representatives[
            representatives["spatial_category"]
            == "low_to_high"
        ]
        .sort_values(
            "mean_delta_fit",
            ascending=False,
        )
    )

    representative_order = (
        high_rep["term_id"].tolist()
        + low_rep["term_id"].tolist()
    )

    rep_profile = mean_profile_df[
        mean_profile_df["term_id"].isin(
            representative_order
        )
    ].copy()

    if len(rep_profile) > 0:
        heatmap_raw = (
            rep_profile
            .pivot(
                index="term_id",
                columns="bin",
                values="score_smooth3_mean",
            )
            .reindex(representative_order)
        )

        row_mean = heatmap_raw.mean(axis=1)
        row_std = (
            heatmap_raw
            .std(axis=1)
            .replace(0, np.nan)
        )

        heatmap_z = (
            heatmap_raw
            .sub(row_mean, axis=0)
            .div(row_std, axis=0)
        )
    else:
        heatmap_raw = pd.DataFrame()
        heatmap_z = pd.DataFrame()

    summary = pd.DataFrame({
        "axis": [axis],
        "region": [region],
        "celltype": [celltype],
        "spatial_terms_before_reduction": [
            len(clustered_all)
        ],
        "representative_programs": [
            len(representatives)
        ],
        "high_to_low_terms_before": [
            (
                stats_df["spatial_category"]
                == "high_to_low"
            ).sum()
        ],
        "high_to_low_representatives": [
            len(high_rep)
        ],
        "low_to_high_terms_before": [
            (
                stats_df["spatial_category"]
                == "low_to_high"
            ).sum()
        ],
        "low_to_high_representatives": [
            len(low_rep)
        ],
    })

    return {
        "clustered_all": clustered_all,
        "representatives": representatives,
        "cluster_summary": cluster_summary,
        "low_pairwise": low_result["pairwise"],
        "high_pairwise": high_result["pairwise"],
        "heatmap_raw": heatmap_raw,
        "heatmap_z": heatmap_z,
        "summary": summary,
    }


## Run redundancy reduction for all ten axis groups


In [ ]:
def run_all_redundancy_reduction(matrix_manifest):
    all_summaries = []
    all_representatives = []

    for _, info in matrix_manifest.iterrows():
        axis = str(info["axis"])
        region = str(info["region"])
        celltype = str(info["celltype"])

        print(
            "\n" + "=" * 80,
            f"\naxis_{axis} | region_{region} | {celltype}",
        )

        result = analyze_redundancy_group(
            axis=axis,
            region=region,
            celltype=celltype,
        )

        group_outdir = (
            REDUNDANCY_DIR
            / f"axis_{axis}"
            / f"region_{region}"
            / f"celltype_{celltype}"
        )
        group_outdir.mkdir(
            parents=True,
            exist_ok=True,
        )

        prefix = (
            f"axis_{axis}_region_{region}_"
            f"celltype_{celltype}"
        )

        result["clustered_all"].to_csv(
            group_outdir
            / f"{prefix}_spatial_terms_redundancy_clusters.csv",
            index=False,
        )
        result["representatives"].to_csv(
            group_outdir
            / f"{prefix}_representative_spatial_terms.csv",
            index=False,
        )
        result["cluster_summary"].to_csv(
            group_outdir
            / f"{prefix}_functional_program_clusters.csv",
            index=False,
        )
        result["low_pairwise"].to_csv(
            group_outdir
            / f"{prefix}_low_to_high_pairwise_similarity.csv",
            index=False,
        )
        result["high_pairwise"].to_csv(
            group_outdir
            / f"{prefix}_high_to_low_pairwise_similarity.csv",
            index=False,
        )

        if not result["heatmap_raw"].empty:
            result["heatmap_raw"].to_csv(
                group_outdir
                / f"{prefix}_representative_term_heatmap_raw.csv"
            )
            result["heatmap_z"].to_csv(
                group_outdir
                / f"{prefix}_representative_term_heatmap_zscore.csv"
            )

        all_summaries.append(
            result["summary"]
        )

        rep = result["representatives"].copy()
        rep["axis"] = axis
        rep["region"] = region
        rep["celltype"] = celltype

        all_representatives.append(rep)

    redundancy_summary = pd.concat(
        all_summaries,
        ignore_index=True,
    )

    all_representative_terms = pd.concat(
        all_representatives,
        ignore_index=True,
    )

    redundancy_summary.to_csv(
        REDUNDANCY_DIR
        / "all_axis_region_celltype_redundancy_summary.csv",
        index=False,
    )

    all_representative_terms.to_csv(
        REDUNDANCY_DIR
        / "all_axis_region_celltype_representative_spatial_terms.csv",
        index=False,
    )

    return (
        redundancy_summary,
        all_representative_terms,
    )


# Run after Stage 3:
#
# redundancy_summary, representative_terms = (
#     run_all_redundancy_reduction(matrix_manifest)
# )
#
# redundancy_summary
